# Minimum-cost network flow: meet every demand at the least shipping cost

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProvenDecisions/optimization-examples/blob/main/notebooks/03-networks-and-routing/minimum-cost-flow.ipynb)

Solved and checked by [ProvenDecisions](https://proven-decisions.ai/examples/minimum-cost-flow?ref=github): the step-by-step walkthrough, the dual or KKT certificate and a version you can change and re-solve are on the example's page.

## The problem

Ship 8 units from node 1 to node 4. Arcs (cost in \$ per unit, capacity in units): 1->2 (2, 5), 1->3 (4, 6), 2->3 (1, 4), 2->4 (6, 4), 3->4 (2, 7). Minimize total cost.

## The model

Each line has a plain-language note.

$$
\begin{aligned}\min\;& 2 x_{12} + 4 x_{13} + x_{23} + 6 x_{24} + 2 x_{34} &&\text{make as small as possible: total shipping cost (\$)}\\ \text{s.t.}\;& x_{12} + x_{13} = 8 &&\text{limit 1: left side must equal 8}\\ \;& x_{12} - x_{23} - x_{24} = 0 &&\text{limit 2: left side must equal 0}\\ \;& x_{13} + x_{23} - x_{34} = 0 &&\text{limit 3: left side must equal 0}\\ \;& x_{24} + x_{34} = 8 &&\text{limit 4: left side must equal 8}\\ \;& x_{12} \le 5 &&\text{units shipped on arc 1->2 (units) is at most 5}\\ \;& x_{13} \le 6 &&\text{units shipped on arc 1->3 (units) is at most 6}\\ \;& x_{23} \le 4 &&\text{units shipped on arc 2->3 (units) is at most 4}\\ \;& x_{24} \le 4 &&\text{units shipped on arc 2->4 (units) is at most 4}\\ \;& x_{34} \le 7 &&\text{units shipped on arc 3->4 (units) is at most 7}\\ \;& x_{12} \ge 0 &&\text{units shipped on arc 1->2 (units) cannot be negative}\\ \;& x_{13} \ge 0 &&\text{units shipped on arc 1->3 (units) cannot be negative}\\ \;& x_{23} \ge 0 &&\text{units shipped on arc 2->3 (units) cannot be negative}\\ \;& x_{24} \ge 0 &&\text{units shipped on arc 2->4 (units) cannot be negative}\\ \;& x_{34} \ge 0 &&\text{units shipped on arc 3->4 (units) cannot be negative}\end{aligned}
$$

## Solve it with cvxpy

Run the cell (in Colab, `cvxpy` is installed by the first line).

In [ ]:
%pip install -q cvxpy highspy

In [1]:
import cvxpy as cp

x12 = cp.Variable(name='x12')  # units shipped on arc 1->2 (units)
x13 = cp.Variable(name='x13')  # units shipped on arc 1->3 (units)
x23 = cp.Variable(name='x23')  # units shipped on arc 2->3 (units)
x24 = cp.Variable(name='x24')  # units shipped on arc 2->4 (units)
x34 = cp.Variable(name='x34')  # units shipped on arc 3->4 (units)

objective = cp.Minimize(2*x12 + 4*x13 + x23 + 6*x24 + 2*x34)
constraints = [
    x12 + x13 == 8,
    x12 - x23 - x24 == 0,
    x13 + x23 - x34 == 0,
    x24 + x34 == 8,
    x12 <= 5,
    x13 <= 6,
    x23 <= 4,
    x24 <= 4,
    x34 <= 7,
    x12 >= 0,
    x13 >= 0,
    x23 >= 0,
    x24 >= 0,
    x34 >= 0,
]
problem = cp.Problem(objective, constraints)
problem.solve(solver='HIGHS')
print(problem.status, problem.value)
print('x12 =', x12.value)
print('x13 =', x13.value)
print('x23 =', x23.value)
print('x24 =', x24.value)
print('x34 =', x34.value)

optimal 46.0
x12 = 5.0
x13 = 3.0
x23 = 4.0
x24 = 1.0
x34 = 7.0


## The checked answer

The smallest objective value is **46**, at x12 = 5, x13 = 3, x23 = 4, x24 = 1, x34 = 7. All 14 constraints are satisfied (violation ≤0e+00). **Exactly proven optimal.**

**Problem class:** [LP](https://proven-decisions.ai/problem-classes?ref=github); **solver:** HIGHS. What makes the answer proven: a [dual or KKT certificate](https://proven-decisions.ai/glossary?ref=github), explained in plain words.

Step-by-step walkthrough, the proof of optimality and more: [Minimum-cost network flow: meet every demand at the least shipping cost](https://proven-decisions.ai/examples/minimum-cost-flow?ref=github)

**Change the numbers and solve it free:** [this model in the ProvenDecisions solver](https://proven-decisions.ai/?try=minimize%202%2Ax12%20%2B%204%2Ax13%20%2B%201%2Ax23%20%2B%206%2Ax24%20%2B%202%2Ax34%0Asubject%20to%0Ax12%20%2B%20x13%20%3D%208%0Ax12%20-%20x23%20-%20x24%20%3D%200%0Ax13%20%2B%20x23%20-%20x34%20%3D%200%0Ax24%20%2B%20x34%20%3D%208%0Ax12%20%3C%3D%205%0Ax13%20%3C%3D%206%0Ax23%20%3C%3D%204%0Ax24%20%3C%3D%204%0Ax34%20%3C%3D%207%0Ax12%2C%20x13%2C%20x23%2C%20x24%2C%20x34%20%3E%3D%200%0A%23%20x12%20%3D%20units%20shipped%20on%20arc%201-%3E2%20%28units%29%0A%23%20x13%20%3D%20units%20shipped%20on%20arc%201-%3E3%20%28units%29%0A%23%20x23%20%3D%20units%20shipped%20on%20arc%202-%3E3%20%28units%29%0A%23%20x24%20%3D%20units%20shipped%20on%20arc%202-%3E4%20%28units%29%0A%23%20x34%20%3D%20units%20shipped%20on%20arc%203-%3E4%20%28units%29%0A%23%20objective%20%3D%20total%20shipping%20cost%20%28%24%29&ref=github)